## Task1

In [0]:
%sql
CREATE OR REPLACE TABLE bronze_orders
TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
AS 
SELECT 
  *, 
  current_timestamp() AS ingestion_timestamp,
  input_file_name() AS source_file_name
FROM read_files(
  '/Volumes/dbacademy/get_started_de/myfiles/day_1.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);


SELECT * FROM bronze_orders LIMIT 5;

Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,_rescued_data,ingestion_timestamp,source_file_name
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0,41.9136,null,2026-07-08T14:01:52.969Z,dbfs:/Volumes/dbacademy/get_started_de/myfiles/day_1.csv
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0,219.582,null,2026-07-08T14:01:52.969Z,dbfs:/Volumes/dbacademy/get_started_de/myfiles/day_1.csv
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0,6.8714,null,2026-07-08T14:01:52.969Z,dbfs:/Volumes/dbacademy/get_started_de/myfiles/day_1.csv
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031,null,2026-07-08T14:01:52.969Z,dbfs:/Volumes/dbacademy/get_started_de/myfiles/day_1.csv
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164,null,2026-07-08T14:01:52.969Z,dbfs:/Volumes/dbacademy/get_started_de/myfiles/day_1.csv


## Task2

In [0]:
%sql
CREATE OR REPLACE TABLE silver_orders AS
WITH deduplicated AS (
  SELECT 
    *,
    ROW_NUMBER() OVER (PARTITION BY `Row ID` ORDER BY ingestion_timestamp DESC) AS row_num
  FROM bronze_orders
)
SELECT 
  `Row ID` AS row_id,
  `Order ID` AS order_id,
  TRIM(`Customer Name`) AS customer_name,
  TRIM(`Segment`) AS segment,
  TRIM(`Country`) AS country,
  TRIM(`City`) AS city,
  TRIM(`State`) AS state,
  `Postal Code` AS postal_code,
  TRIM(`Region`) AS region,
  `Product ID` AS product_id,
  TRIM(`Category`) AS category,
  TRIM(`Sub-Category`) AS sub_category,
  TRIM(`Product Name`) AS product_name,
  TRIM(`Ship Mode`) AS ship_mode,

  try_cast(`Order Date` AS DATE) AS order_date,
  try_cast(`Ship Date` AS DATE) AS ship_date,
  try_cast(`Sales` AS DOUBLE) AS sales,
  try_cast(`Quantity` AS INT) AS quantity,
  try_cast(`Discount` AS DOUBLE) AS discount,
  try_cast(`Profit` AS DOUBLE) AS profit,
  
  ingestion_timestamp,
  source_file_name
FROM deduplicated
WHERE row_num = 1                       
  AND try_cast(`Sales` AS DOUBLE) >= 0.0                      
  AND try_cast(`Quantity` AS INT) > 0                    
  AND try_cast(`Ship Date` AS DATE) >= try_cast(`Order Date` AS DATE);


SELECT * FROM silver_orders LIMIT 5;

row_id,order_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,ship_mode,order_date,ship_date,sales,quantity,discount,profit,ingestion_timestamp,source_file_name
1,CA-2016-152156,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,Second Class,2016-11-08,2016-11-11,261.96,2,0.0,41.9136,2026-07-08T14:01:52.969Z,dbfs:/Volumes/dbacademy/get_started_de/myfiles/day_1.csv
2,CA-2016-152156,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",Second Class,2016-11-08,2016-11-11,731.94,3,0.0,219.582,2026-07-08T14:01:52.969Z,dbfs:/Volumes/dbacademy/get_started_de/myfiles/day_1.csv
3,CA-2016-138688,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,Second Class,2016-06-12,2016-06-16,14.62,2,0.0,6.8714,2026-07-08T14:01:52.969Z,dbfs:/Volumes/dbacademy/get_started_de/myfiles/day_1.csv
4,US-2015-108966,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,Standard Class,2015-10-11,2015-10-18,957.5775,5,0.45,-383.031,2026-07-08T14:01:52.969Z,dbfs:/Volumes/dbacademy/get_started_de/myfiles/day_1.csv
5,US-2015-108966,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,Standard Class,2015-10-11,2015-10-18,22.368,2,0.2,2.5164,2026-07-08T14:01:52.969Z,dbfs:/Volumes/dbacademy/get_started_de/myfiles/day_1.csv


In [0]:
%sql
CREATE OR REPLACE TABLE silver_rejected_orders AS
SELECT 
  `Row ID` AS row_id,
  `Order ID` AS order_id,
  `Customer Name` AS customer_name,
  `Sales` AS sales,
  `Quantity` AS quantity,
  `Order Date` AS order_date,
  `Ship Date` AS ship_date,
  ingestion_timestamp,
  source_file_name
FROM bronze_orders
WHERE try_cast(`Sales` AS DOUBLE) IS NULL 
   OR try_cast(`Quantity` AS INT) IS NULL
   OR try_cast(`Order Date` AS DATE) IS NULL
   OR try_cast(`Ship Date` AS DATE) IS NULL
   OR try_cast(`Sales` AS DOUBLE) < 0.0 
   OR try_cast(`Quantity` AS INT) <= 0 
   OR try_cast(`Ship Date` AS DATE) < try_cast(`Order Date` AS DATE);

SELECT * FROM silver_rejected_orders LIMIT 5;

row_id,order_id,customer_name,sales,quantity,order_date,ship_date,ingestion_timestamp,source_file_name
33,US-2015-150630,Tracy Blumstein,"Executive Red""",6.858,2015-09-17,2015-09-21,2026-07-08T14:01:52.969Z,dbfs:/Volumes/dbacademy/get_started_de/myfiles/day_1.csv
77,US-2017-118038,Ken Brennan,"Black""",9.708,2017-12-09,2017-12-11,2026-07-08T14:01:52.969Z,dbfs:/Volumes/dbacademy/get_started_de/myfiles/day_1.csv
106,US-2015-156867,Lena Cacioppo,Black,"1040 sheets""",2015-11-13,2015-11-17,2026-07-08T14:01:52.969Z,dbfs:/Volumes/dbacademy/get_started_de/myfiles/day_1.csv
199,US-2017-124303,Fred Hopkins,"Light Blue""",2.946,2017-07-06,2017-07-13,2026-07-08T14:01:52.969Z,dbfs:/Volumes/dbacademy/get_started_de/myfiles/day_1.csv
256,US-2015-159982,Dan Reichenbach,"Black""",82.368,2015-11-28,2015-12-04,2026-07-08T14:01:52.969Z,dbfs:/Volumes/dbacademy/get_started_de/myfiles/day_1.csv


## Task 3

In [0]:
%sql
MERGE INTO silver_orders AS target
USING (
  WITH raw_day3 AS (
    SELECT 
      *, 
      current_timestamp() AS ingestion_timestamp,
      _metadata.file_path AS source_file_name
    FROM read_files(
      '/Volumes/dbacademy/get_started_de/myfiles/day_3.csv', 
      format => 'csv',
      header => true,
      inferSchema => true
    )
  ),
  deduplicated AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY `Row ID` ORDER BY ingestion_timestamp DESC) AS row_num
    FROM raw_day3
  )
  SELECT 
    `Row ID` AS row_id,
    `Order ID` AS order_id,
    TRIM(`Customer Name`) AS customer_name,
    TRIM(`Segment`) AS segment,
    TRIM(`Country`) AS country,
    TRIM(`City`) AS city,
    TRIM(`State`) AS state,
    `Postal Code` AS postal_code,
    TRIM(`Region`) AS region,
    `Product ID` AS product_id,
    TRIM(`Category`) AS category,
    TRIM(`Sub-Category`) AS sub_category,
    TRIM(`Product Name`) AS product_name,
    TRIM(`Ship Mode`) AS ship_mode,
    try_cast(`Order Date` AS DATE) AS order_date,
    try_cast(`Ship Date` AS DATE) AS ship_date,
    try_cast(`Sales` AS DOUBLE) AS sales,
    try_cast(`Quantity` AS INT) AS quantity,
    try_cast(`Discount` AS DOUBLE) AS discount,
    try_cast(`Profit` AS DOUBLE) AS profit,
    ingestion_timestamp,
    source_file_name
  FROM deduplicated
  WHERE row_num = 1
    AND try_cast(`Sales` AS DOUBLE) >= 0.0
    AND try_cast(`Quantity` AS INT) > 0
    AND try_cast(`Ship Date` AS DATE) >= try_cast(`Order Date` AS DATE)
) AS source
ON target.row_id = source.row_id

WHEN MATCHED THEN
  UPDATE SET 
    target.order_id = source.order_id,
    target.customer_name = source.customer_name,
    target.segment = source.segment,
    target.country = source.country,
    target.city = source.city,
    target.state = source.state,
    target.postal_code = source.postal_code,
    target.region = source.region,
    target.product_id = source.product_id,
    target.category = source.category,
    target.sub_category = source.sub_category,
    target.product_name = source.product_name,
    target.ship_mode = source.ship_mode,
    target.order_date = source.order_date,
    target.ship_date = source.ship_date,
    target.sales = source.sales,
    target.quantity = source.quantity,
    target.discount = source.discount,
    target.profit = source.profit,
    target.ingestion_timestamp = source.ingestion_timestamp,
    target.source_file_name = source.source_file_name

WHEN NOT MATCHED THEN
  INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3227,0,0,3227


In [0]:
%sql
SELECT COUNT(*) AS rows_count 
FROM silver_orders;

rows_count
6468


## Task 4

In [0]:
%sql
CREATE OR REPLACE TABLE gold_sales_daily AS
SELECT 
  order_date AS date, 
  SUM(sales) AS revenue
FROM silver_orders
GROUP BY order_date
ORDER BY date;

select * from gold_sales_daily
LIMIT 5;

date,revenue
2014-01-04,288.06
2014-01-05,19.536
2014-01-06,4387.66
2014-01-07,87.15799999999999
2014-01-09,40.544


In [0]:
%sql
CREATE OR REPLACE TABLE gold_sales_region AS
SELECT 
  region, 
  SUM(sales) AS revenue
FROM silver_orders
GROUP BY region;

select * from gold_sales_region
LIMIT 5;

region,revenue
East,440372.7680000003
South,271857.848
Central,340828.13979999954
West,486469.5680000002


In [0]:
%sql
CREATE OR REPLACE TABLE gold_sales_category AS
SELECT 
  category, 
  SUM(sales) AS revenue
FROM silver_orders
GROUP BY category;

select * from gold_sales_category;

category,revenue
Office Supplies,467264.3189999995
Technology,577739.0939999997
Furniture,494524.9108000004


In [0]:
%sql
CREATE OR REPLACE TABLE gold_customer_metrics AS
SELECT 
  customer_name AS customer, 
  SUM(sales) AS revenue, 
  COUNT(DISTINCT order_id) AS orders
FROM silver_orders
GROUP BY customer_name;

select * from gold_customer_metrics
LIMIT 5;

customer,revenue,orders
Ken Brennan,764.514,6
Chloris Kastensmidt,1315.828,8
Julie Prescott,1209.734,7
John Dryer,278.74399999999997,3
Cynthia Delaney,1860.732,5


In [0]:
%sql
SELECT 
  order_id, 
  COUNT(*) AS rows_count
FROM silver_orders
GROUP BY order_id
ORDER BY rows_count DESC
LIMIT 5;

order_id,rows_count
CA-2017-157987,12
CA-2016-165330,10
US-2016-108504,10
CA-2015-131338,10
US-2016-114013,9


## Task5

In [0]:
%sql
WITH ranked_products AS (
  SELECT 
    product_name, 
    SUM(sales) AS revenue,
    ROW_NUMBER() OVER (ORDER BY SUM(sales) DESC) AS rank
  FROM silver_orders
  GROUP BY product_name
)
SELECT rank, product_name, revenue
FROM ranked_products
WHERE rank <= 5
LIMIT 5;;

rank,product_name,revenue
1,Canon imageCLASS 2200 Advanced Copier,42699.878
2,Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind,25928.196
3,Cisco TelePresence System EX90 Videoconferencing Unit,22638.48
4,HON 5400 Series Task Chairs for Big and Tall,18365.676
5,GBC Ibimaster 500 Manual ProClick Binding System,17426.442


In [0]:
%sql
SELECT 
  region, 
  SUM(sales) AS revenue
FROM silver_orders
GROUP BY region
ORDER BY revenue DESC
LIMIT 1;

region,revenue
West,486469.5680000002


In [0]:
%sql
WITH monthly_sales AS (
  SELECT 
    DATE_TRUNC('month', order_date) AS month,
    SUM(sales) AS revenue
  FROM silver_orders
  GROUP BY DATE_TRUNC('month', order_date)
)
SELECT 
  month,
  revenue,
  LAG(revenue) OVER (ORDER BY month) AS previous_month_revenue,
  revenue - LAG(revenue) OVER (ORDER BY month) AS month_over_month_change
FROM monthly_sales
ORDER BY month
LIMIT 5;

month,revenue,previous_month_revenue,month_over_month_change
2014-01-01T00:00:00.000Z,10270.68,null,null
2014-02-01T00:00:00.000Z,2453.58,10270.68,-7817.1
2014-03-01T00:00:00.000Z,50276.666,2453.58,47823.085999999996
2014-04-01T00:00:00.000Z,15797.735,50276.666,-34478.931
2014-05-01T00:00:00.000Z,20232.403000000006,15797.735,4434.668000000005


In [0]:
%sql
WITH daily_sales AS (
  SELECT 
    order_date, 
    SUM(sales) AS daily_revenue
  FROM silver_orders
  GROUP BY order_date
)
SELECT 
  order_date,
  daily_revenue,
  SUM(daily_revenue) OVER (ORDER BY order_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total
FROM daily_sales
ORDER BY order_date
LIMIT 5;

order_date,daily_revenue,running_total
2014-01-04,288.06,288.06
2014-01-05,19.536,307.596
2014-01-06,4387.66,4695.255999999999
2014-01-07,87.15799999999999,4782.414
2014-01-09,40.544,4822.958


In [0]:
%sql
WITH product_region_revenue AS (
  SELECT 
    region, 
    product_name, 
    SUM(sales) AS revenue
  FROM silver_orders
  GROUP BY region, product_name
),
ranked_in_region AS (
  SELECT 
    region, 
    product_name, 
    revenue,
    ROW_NUMBER() OVER (PARTITION BY region ORDER BY revenue DESC) AS rank
  FROM product_region_revenue
)
SELECT region, product_name, revenue
FROM ranked_in_region
WHERE rank = 1
LIMIT 5;

region,product_name,revenue
Central,Canon imageCLASS 2200 Advanced Copier,17499.95
East,Canon imageCLASS 2200 Advanced Copier,11199.968
South,Cisco TelePresence System EX90 Videoconferencing Unit,22638.48
West,Canon imageCLASS 2200 Advanced Copier,13999.96


## Task6
Calculate what percentage of regional revenue a specific receipt generated.

In [0]:
EXPLAIN
SELECT /*+ MERGE(r) */
  o.order_id,
  o.customer_name,
  o.region,
  o.sales AS order_amount,
  r.revenue AS total_region_revenue,
  ROUND((o.sales / r.revenue) * 100, 4) AS contribution_percentage
FROM silver_orders o
JOIN gold_sales_region r ON o.region = r.region;

plan
"== Physical Plan == AdaptiveSparkPlan isFinalPlan=false +- == Initial Plan == Project [order_id#21454, customer_name#21455, region#21461, sales#21469 AS order_amount#21421, revenue#21478 AS total_region_revenue#21422, round(((sales#21469 / revenue#21478) * 100.0), 4) AS contribution_percentage#21423] +- SortMergeJoin [region#21461], [region#21477], Inner :- ColumnarToRow : +- PhotonResultStage : +- PhotonSort [region#21461 ASC NULLS FIRST] : +- PhotonShuffleExchangeSource : +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#15186] : +- PhotonShuffleExchangeSink hashpartitioning(region#21461, 16) : +- PhotonScan parquet workspace.default.silver_orders[order_id#21454,customer_name#21455,region#21461,sales#21469] DataFilters: [isnotnull(region#21461)], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://dbstorage-prod-6p1dq/uc/c4b8dc16-67f7-4112-8826-1d34a221e81b..., OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct, RequiredDataFilters: [isnotnull(region#21461)] +- ColumnarToRow +- PhotonResultStage +- PhotonSort [region#21477 ASC NULLS FIRST] +- PhotonShuffleExchangeSource +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#15194] +- PhotonShuffleExchangeSink hashpartitioning(region#21477, 16) +- PhotonScan parquet workspace.default.gold_sales_region[region#21477,revenue#21478] DataFilters: [isnotnull(region#21477)], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://dbstorage-prod-6p1dq/uc/c4b8dc16-67f7-4112-8826-1d34a221e81b..., OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct, RequiredDataFilters: [isnotnull(region#21477)] == Photon Explanation == Photon does not fully support the query because: Unsupported node: SortMergeJoin [region#21461], [region#21477], Inner. Reference node: SortMergeJoin [region#21461], [region#21477], Inner == Optimizer Statistics (table names per statistics state) == missing = partial = full = gold_sales_region, silver_orders"


In [0]:
EXPLAIN
SELECT /*+ BRODCAST(r) */
  o.order_id,
  o.customer_name,
  o.region,
  o.sales AS order_amount,
  r.revenue AS total_region_revenue,
  ROUND((o.sales / r.revenue) * 100, 4) AS contribution_percentage
FROM silver_orders o
JOIN gold_sales_region r ON o.region = r.region;

plan
"== Physical Plan == AdaptiveSparkPlan isFinalPlan=false +- == Initial Plan == PhotonResultStage +- PhotonColumnarToRow +- PhotonProject [order_id#21531, customer_name#21532, region#21538, sales#21546 AS order_amount#21496, revenue#21553 AS total_region_revenue#21497, round(((sales#21546 / revenue#21553) * 100.0), 4) AS contribution_percentage#21498] +- PhotonBroadcastHashJoin [region#21538], [region#21552], Inner, BuildRight, false, true :- PhotonScan parquet workspace.default.silver_orders[order_id#21531,customer_name#21532,region#21538,sales#21546] DataFilters: [isnotnull(region#21538)], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://dbstorage-prod-6p1dq/uc/c4b8dc16-67f7-4112-8826-1d34a221e81b..., OptionalDataFilters: [hashedrelationcontains(region#21538)], PartitionFilters: [], ReadSchema: struct, RequiredDataFilters: [isnotnull(region#21538)] +- PhotonShuffleExchangeSource +- PhotonShuffleMapStage EXECUTOR_BROADCAST, [id=#15291] +- PhotonShuffleExchangeSink SinglePartition +- PhotonScan parquet workspace.default.gold_sales_region[region#21552,revenue#21553] DataFilters: [isnotnull(region#21552)], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://dbstorage-prod-6p1dq/uc/c4b8dc16-67f7-4112-8826-1d34a221e81b..., OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct, RequiredDataFilters: [isnotnull(region#21552)] == Photon Explanation == The query is fully supported by Photon. == Optimizer Statistics (table names per statistics state) == missing = partial = full = gold_sales_region, silver_orders"
